# Concevoir l'agent du service client : outils, MCP, ReAct et planification

Le fil rouge de la formation : l'agent du service client de Biclou & Cie, qui lit le message d'un client, appelle les outils du magasin (la fiche d'une commande, le stock, les règles du magasin, un ticket, le transfert à un conseiller), puis répond selon la charte. Ce TP le construit avec un petit modèle open source, Qwen3-1.7B : d'abord sans outil, pour voir ce qu'un prompt seul invente ; puis ses outils, servis par un serveur MCP ; sa boucle ReAct ; une variante qui planifie avant d'agir ; une relecture de ses réponses ; sa mémoire. Il finit par relever ses erreurs, d'où partira le TP suivant pour constituer son jeu d'entraînement.

1. Un prompt seul
2. Les outils, servis par MCP
3. La boucle ReAct
4. Plan & Execute
5. Réflexion : relire et corriger
6. État et mémoire
7. Relever les erreurs de l'agent

Ce notebook demande un GPU : *Exécution > Modifier le type d'exécution > GPU T4*, sur lequel il tient en une vingtaine de minutes (une estimation). Sans GPU, il tourne quand même, en plus petit : Qwen3-0.6B au lieu de Qwen3-1.7B, et moins de demandes à chaque exercice : 40 minutes environ sur deux processeurs, comme ceux de Colab. Ce petit modèle, le raisonnement éteint comme pour l'agent du cours, n'appelle presque jamais ses outils : il répond sans chercher (exercices 3, 4 et 7). C'est l'erreur d'où partent les deux TP suivants, qui constituent un jeu d'entraînement, puis affinent le modèle sur lui.

## Données et modèle

Les données de l'agent, dans le dossier `agent/` du jeu `bike-shop-support` : `demandes.jsonl`, l'historique annoté, 1 300 demandes de clients avec les appels que l'agent devait faire, leurs résultats et la réponse ; `commandes.csv` et `stock.csv`, ce que lisent les outils ; `outils.json`, leurs schémas de référence. Et `charte.md`, la charte du service client, dont la partie « Politique » est ce que cherche l'outil `chercher_politique`. Le dossier a aussi un jeu d'évaluation figé, `eval.jsonl`, que ce TP ne télécharge pas : les derniers chapitres y comparent les modèles, et rien ne doit être réglé sur lui.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/agent/demandes.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/agent/commandes.csv
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/agent/stock.csv
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/agent/outils.json
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/charte.md
%pip install -q "mcp>=2,<3"

In [ ]:
import copy
import json
import random
import re
import sys
import time
from datetime import date, timedelta
from pathlib import Path
from typing import Literal

import jsonschema
import pandas as pd
import torch
from mcp import Client, StdioServerParameters
from pydantic import BaseModel, ValidationError
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache

GPU = torch.cuda.is_available()
DTYPE = torch.float16 if GPU else torch.float32  # sur CPU, float32 : bien plus rapide
DEVICE = "cuda" if GPU else "cpu"
MODELE_AGENT = "Qwen/Qwen3-1.7B" if GPU else "Qwen/Qwen3-0.6B"
# Sans GPU, en plus petit : les suivis de commande des exercices 1 et 3 (un seul avec
# pensée), les demandes composées de l'exercice 4, et les demandes par comportement
# attendu de l'exercice 7.
N_DIRECTES = 3 if GPU else 2
N_PENSEE = 3 if GPU else 1
N_COMPOSEES = 6 if GPU else 2
N_PAR_CLASSE = 4 if GPU else 1
print(MODELE_AGENT, "sur GPU" if GPU else "sans GPU, en plus petit")

demandes = pd.read_json("demandes.jsonl", lines=True)
with open("outils.json", encoding="utf-8") as f:
  outils_reference = json.load(f)
with open("charte.md", encoding="utf-8") as f:
  charte = f.read()
print(len(demandes), "demandes dans l'historique")


def classe(demande) -> str:
  """Le comportement attendu : les outils, dans l'ordre, ou le cas sans outil."""
  noms = [a["name"] for a in demande["appels_attendus"]]
  if noms:
    return "+".join(noms)
  cas = "sans_numero" if demande["cas"] == "sans_numero" else demande["scenario"]
  return f"aucun:{cas}"


demandes["classe"] = demandes.apply(classe, axis=1)
demandes["classe"].value_counts()

Le système du magasin, `biclou.py` (cellule ci-dessous) : ses données et les fonctions que les outils de l'agent appelleront. La fiche d'une commande, le stock d'un vélo, la recherche par mots-clés dans les règles du magasin (la recherche documentaire de l'agent, un outil parmi d'autres), la création d'un ticket et le transfert à un conseiller. Le serveur MCP de l'exercice 2 les importe ; le notebook aussi, pour lire une fiche.

In [ ]:
%%writefile biclou.py
"""Le système de Biclou & Cie : ses données, et ce que font les outils de l'agent."""

import re
import unicodedata

import pandas as pd

commandes = pd.read_csv("commandes.csv", dtype=str, keep_default_na=False)
commandes = commandes.set_index("numero")
stock = pd.read_csv("stock.csv", keep_default_na=False)
with open("charte.md", encoding="utf-8") as f:
  charte = f.read()

CATEGORIES = [
  "information",
  "livraison",
  "reparation",
  "retour",
  "location",
  "facturation",
  "compte",
]
PRIORITES = ["basse", "normale", "haute"]
RAISONS = ["securite", "colis_perdu", "paiement", "accident_location", "vol_location"]
MODELES = list(dict.fromkeys(stock["modele"]))
TAILLES = list(dict.fromkeys(stock["taille"]))
MAGASINS = list(dict.fromkeys(stock["magasin"]))
RUBRIQUES = {
  "Retours": "retours",
  "Livraison": "livraison",
  "Garantie": "garantie",
  "Atelier": "atelier",
  "Location": "location",
  "Facturation": "facturation",
  "Compte": "compte",
  "Magasins": "magasins",
  "Tailles de cadre": "tailles",
}


def lit_politique(charte: str) -> dict[str, list[str]]:
  """Les phrases de chaque rubrique de la partie « Politique » de la charte."""
  texte = charte.replace("\xa0", " ").split("## Politique", 1)[1]
  puces = re.findall(r"^- \*\*(.+?)\*\* : (.+?)(?=^- |\Z)", texte, re.M | re.S)
  return {
    RUBRIQUES[titre]: re.split(
      r"(?<=\.) (?=[A-ZÀ-Ý])", f"{titre} : {' '.join(corps.split())}"
    )
    for titre, corps in puces
  }


PASSAGES = lit_politique(charte)
THEMES = list(PASSAGES)


def racines(texte: str) -> set[str]:
  """Les mots de 4 lettres ou plus, comparés par leurs 5 premières lettres."""
  texte = unicodedata.normalize("NFKD", texte.lower())
  texte = "".join(c for c in texte if not unicodedata.combining(c))
  return {mot[:5] for mot in re.findall(r"[a-z]+", texte) if len(mot) >= 4}


def chercher_commande(numero: str) -> dict:
  """La fiche de la commande, sans ses champs vides, ou une erreur."""
  if not re.fullmatch(r"CMD-\d{6}", numero):
    return {"erreur": "numéro invalide : CMD- suivi de 6 chiffres"}
  if numero not in commandes.index:
    return {"erreur": f"commande introuvable : {numero}"}
  fiche = {"numero": numero, **commandes.loc[numero].to_dict()}
  fiche = {cle: valeur for cle, valeur in fiche.items() if valeur != ""}
  fiche["montant"] = float(fiche["montant"])
  fiche["paiements"] = int(fiche["paiements"])
  return fiche


def verifier_stock(modele: str, taille: str, magasin: str) -> dict:
  """La quantité en stock et, en rupture, la date du prochain réassort."""
  ligne = stock[
    (stock["modele"] == modele)
    & (stock["taille"] == taille)
    & (stock["magasin"] == magasin)
  ]
  if ligne.empty:
    return {"erreur": "modèle, taille ou magasin inconnu"}
  quantite = int(ligne["quantite"].iloc[0])
  resultat = dict(modele=modele, taille=taille, magasin=magasin, quantite=quantite)
  if quantite == 0:
    resultat["reassort"] = ligne["reassort"].iloc[0]
  return resultat


def chercher_politique(theme: str, question: str) -> dict:
  """Les trois passages de la rubrique les plus proches de la question."""
  if theme not in PASSAGES:
    return {"erreur": f"rubrique inconnue : {theme}"}
  mots = racines(question)
  tries = sorted(
    enumerate(PASSAGES[theme]), key=lambda p: (-len(mots & racines(p[1])), p[0])
  )
  return {"theme": theme, "passages": [passage for _, passage in tries[:3]]}


compteurs = {"ticket": 30000, "dossier": 5000}
EQUIPES = {
  "securite": ("atelier", "aujourd'hui"),
  "colis_perdu": ("livraison", "sous 24 h"),
  "paiement": ("comptabilite", "sous 24 h"),
  "accident_location": ("location", "sous 24 h"),
  "vol_location": ("location", "sous 24 h"),
}


def creer_ticket(categorie: str, priorite: str, resume: str) -> dict:
  """Le numéro du ticket créé."""
  if categorie not in CATEGORIES or priorite not in PRIORITES:
    return {"erreur": "catégorie ou priorité inconnue"}
  compteurs["ticket"] += 1
  return {"ticket": f"T-{compteurs['ticket']}", "statut": "ouvert"}


def escalader(raison: str, resume: str) -> dict:
  """Le numéro du dossier transmis et l'équipe qui le prend en charge."""
  if raison not in EQUIPES:
    return {"erreur": f"raison inconnue : {raison}"}
  compteurs["dossier"] += 1
  equipe, recontact = EQUIPES[raison]
  return {
    "dossier": f"ESC-{compteurs['dossier']}",
    "equipe": equipe,
    "recontact": recontact,
  }

La consigne de l'agent, la même pour toute la formation : son rôle, quand appeler chaque outil, et les règles de réponse de la charte. Et la génération (cellule ci-dessous) : `genere_tout`, la suite de chaque conversation, par lots sur GPU, une à une sans GPU, où `prefixe` calcule une fois le début commun des prompts, le message `system` et les schémas des outils.

In [ ]:
import biclou

CONSIGNES_AGENT = """Tu es l'agent du service client de Biclou & Cie, qui vend, répare et loue des vélos en ligne et dans six magasins (Lille, Amiens, Nice, Marseille, Nantes, Rennes). Pour chaque message de client, appelle les outils utiles, un à la fois, puis réponds au client.

Quand appeler un outil :
- un numéro de commande (CMD- et 6 chiffres) dans le message : chercher_commande d'abord ;
- une question sur les règles du magasin (horaires, tailles, garantie, atelier, retours, livraison, location, factures, compte) : chercher_politique ;
- la disponibilité d'un vélo dans un magasin : verifier_stock ;
- un problème que l'équipe doit suivre (colis abîmé, panne, réglage, réduction oubliée, suppression de compte) : creer_ticket ;
- la sécurité (freins, batterie), un colis perdu, un double prélèvement, un accident ou un vol pendant une location : escalader.
Sans outil : un suivi de commande sans son numéro (demande-le), un remerciement, une question sans rapport avec le magasin, une demande à refuser.

Ta réponse suit la charte : vouvoiement, « Bonjour, », ou « Bonjour Madame » ou « Bonjour Monsieur » et le nom de famille qui signe le message, 40 à 120 mots, une prochaine étape claire, aucun remboursement promis avant vérification, « Bonne journée, » puis « L'équipe Biclou & Cie » sur sa propre ligne, et en anglais pour un message en anglais."""


def charge(nom: str):
  return AutoModelForCausalLM.from_pretrained(nom, dtype=DTYPE, device_map=DEVICE)


def prefixe(modele, tok, systeme: str, outils: list | None = None):
  """Sans GPU, le cache du début commun des prompts (le message system et les
  outils), calculé une fois : chaque génération repart d'une copie, comme le fait un
  serveur d'inférence. Sur GPU, None : les lots vont assez vite sans."""
  if GPU:
    return None
  debut = tok.apply_chat_template(
    [dict(role="system", content=systeme)], tools=outils, tokenize=False
  )
  cache = DynamicCache(config=modele.config)
  with torch.no_grad():
    modele(**tok(debut, return_tensors="pt").to(DEVICE), past_key_values=cache)
  return cache


def genere_tout(
  modele,
  tok,
  conversations,
  outils=None,
  cache=None,
  max_tokens=96,
  pense=False,
  **options,
):
  """La suite de chaque conversation : par lots de 8 sur GPU, une à une sans GPU, à
  partir d'une copie de `cache`. Sans échantillonnage, sauf `options`. Avec
  `pense=True`, Qwen3 raisonne d'abord, dans un bloc <think>."""
  options = options or dict(do_sample=False, temperature=None, top_p=None, top_k=None)
  prompts = [
    tok.apply_chat_template(
      c, tools=outils, tokenize=False, add_generation_prompt=True, enable_thinking=pense
    )
    for c in conversations
  ]
  taille_lot = 8 if GPU else 1
  textes = []
  for debut in range(0, len(prompts), taille_lot):
    entrees = tok(
      prompts[debut : debut + taille_lot], return_tensors="pt", padding=True
    )
    copie = copy.deepcopy(cache) if cache is not None else None
    with torch.no_grad():
      sorties = modele.generate(
        **entrees.to(DEVICE),
        past_key_values=copie,
        max_new_tokens=max_tokens,
        **options,
      )
    textes += tok.batch_decode(
      sorties[:, entrees.input_ids.shape[1] :], skip_special_tokens=True
    )
  return textes


def nettoie(texte: str) -> str:
  """Sans les espaces de fin de ligne que le modèle laisse parfois (« Bonne journée,  »)."""
  return "\n".join(ligne.rstrip() for ligne in texte.strip().splitlines())


tokenizer = AutoTokenizer.from_pretrained(MODELE_AGENT)
tokenizer.padding_side = "left"
modele = charge(MODELE_AGENT)

## 1. Un prompt seul

Avant tout outil, le modèle seul, avec un prompt direct : `CONSIGNES_DIRECTES`, la charte entière en message `system`, puis le message du client. Écrivez `reponses_directes(messages)`, la réponse du modèle à chacun des messages, avec `genere_tout` et 200 tokens au plus ; sans GPU, `prefixe` calcule une fois le début commun des prompts, cette consigne. Puis, pour chaque demande de `directes`, des suivis de commande de l'historique qui citent leur numéro, affichez le message, la réponse et la fiche de la commande (`biclou.chercher_commande`). Que dit la réponse de la commande (son statut, une date, un délai, un transporteur) que la fiche dément, ou que rien n'a pu lui apprendre ?

In [ ]:
CONSIGNES_DIRECTES = (
  "Tu es l'agent du service client de Biclou & Cie. Réponds au message du client "
  "en suivant cette charte.\n\n" + charte
)
suivis = demandes[
  demandes["scenario"].isin(["colis_retard", "colis_perdu", "remboursement_attente"])
  & (demandes["cas"] == "nominal")
  & demandes["appels_attendus"].map(
    lambda appels: bool(appels) and appels[0]["name"] == "chercher_commande"
  )
]
directes = (
  suivis.sample(frac=1, random_state=0)
  .groupby("scenario")
  .head(1)
  .head(N_DIRECTES)
  .reset_index(drop=True)
)
print(directes[["scenario", "message"]])

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Une conversation : `[dict(role="system", content=CONSIGNES_DIRECTES), dict(role="user", content=message)]`
- `genere_tout(modele, tokenizer, conversations, cache=prefixe(modele, tokenizer, CONSIGNES_DIRECTES), max_tokens=200)` : une réponse par conversation, sans outils (`outils=None`) ; `nettoie` ôte les espaces autour
- `re.search(r"CMD-\d{6}", message)[0]` : le numéro que cite le message

```python
def reponses_directes(messages: list[str]) -> list[str]:
  conversations = [
    [dict(role="system", content=CONSIGNES_DIRECTES), dict(role="user", content=m)]
    for m in messages
  ]
  cache = prefixe(modele, tokenizer, ...)
  textes = genere_tout(modele, tokenizer, conversations, cache=cache, max_tokens=...)
  return [nettoie(texte) for texte in textes]


debut = time.time()
directes["reponse_directe"] = reponses_directes(list(directes["message"]))
print(f"{len(directes)} réponses en {time.time() - debut:.0f} s")
for d in directes.itertuples():
  numero = re.search(r"CMD-\d{6}", d.message)[0]
  print(f"\n--- {d.message}\n\n{d.reponse_directe}\n")
  print("Fiche :", ...)
```

### Solution

In [ ]:
def reponses_directes(messages: list[str]) -> list[str]:
  conversations = [
    [dict(role="system", content=CONSIGNES_DIRECTES), dict(role="user", content=m)]
    for m in messages
  ]
  cache = prefixe(modele, tokenizer, CONSIGNES_DIRECTES)
  textes = genere_tout(modele, tokenizer, conversations, cache=cache, max_tokens=200)
  return [nettoie(texte) for texte in textes]


debut = time.time()
directes["reponse_directe"] = reponses_directes(list(directes["message"]))
print(f"{len(directes)} réponses en {time.time() - debut:.0f} s")
for d in directes.itertuples():
  numero = re.search(r"CMD-\d{6}", d.message)[0]
  print(f"\n--- {d.message}\n\n{d.reponse_directe}\n")
  print("Fiche :", biclou.chercher_commande(numero))

# Sans la fiche, tout ce que la réponse dit de la commande est deviné : le modèle n'a
# rien lu d'autre que le message.

## 2. Les outils, servis par MCP

Pour un modèle, un outil est un schéma JSON : son nom, sa description, ses arguments et leurs types. MCP (*Model Context Protocol*) sépare les outils de l'agent : un serveur les déclare et les exécute, et n'importe quel client MCP (cet agent, un assistant de bureau, un éditeur de code) les liste et les appelle sans connaître leur code. Le SDK Python `mcp` tire le schéma d'un outil de la fonction qui le déclare : son nom, sa docstring, ses annotations de type.

### 2.1 Le serveur

Transformez les fonctions de `biclou` en outils : écrivez `serveur_biclou.py`, le serveur MCP `biclou`, dans une cellule qui commence par `%%writefile serveur_biclou.py`. Un `MCPServer`, et cinq outils déclarés par `@serveur.tool()`, chacun appelant la fonction de même nom de `biclou`, dans cet ordre : `chercher_commande`, `verifier_stock`, `chercher_politique`, `creer_ticket`, `escalader`. Leurs descriptions, et celles de leurs arguments, sont celles d'`outils_reference` (`outils.json`, affiché ci-dessous) ; les valeurs possibles des arguments à liste fermée sont dans `biclou.MODELES`, `TAILLES`, `MAGASINS`, `THEMES`, `CATEGORIES`, `PRIORITES` et `RAISONS`. Le serveur se lance en stdio (`serveur.run()`) quand on exécute le fichier.

**Documentation utile**

- [Les outils d'un serveur MCP](https://py.sdk.modelcontextprotocol.io/servers/tools/)
- [`typing.Annotated`](https://docs.python.org/3/library/typing.html#typing.Annotated)
- [`pydantic.Field`](https://docs.pydantic.dev/latest/concepts/fields/)

In [ ]:
for outil in outils_reference:
  fonction = outil["function"]
  print(fonction["name"], ":", fonction["description"])
  for nom, argument in fonction["parameters"]["properties"].items():
    valeurs = f" ({len(argument['enum'])} valeurs)" if "enum" in argument else ""
    print(f"  {nom} : {argument['description']}{valeurs}")

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- La docstring entière est la description de l'outil : une phrase, celle d'`outils.json`. Le SDK ne lit pas de section `Args:` ; la description d'un argument s'écrit dans son annotation : `Annotated[str, Field(description="...")]`
- `Literal[tuple(biclou.MODELES)]` : l'énumération des valeurs de `MODELES` ; `Annotated[Literal[tuple(biclou.MODELES)], Field(description="...")]` : l'énumération et la description
- `MCPServer("biclou", log_level="WARNING")` : le serveur ne journalise que ses avertissements
- `if __name__ == "__main__": serveur.run()` : le serveur ne se lance que quand on exécute le fichier, pas quand on l'importe

```python
%%writefile serveur_biclou.py
"""Le serveur MCP des outils de Biclou & Cie, en stdio."""

from typing import Annotated, Literal

import biclou
from mcp.server.mcpserver import MCPServer
from pydantic import Field

serveur = MCPServer("biclou", log_level="WARNING")


@serveur.tool()
def chercher_commande(
  numero: Annotated[
    str, Field(description="le numéro de commande, CMD- suivi de 6 chiffres")
  ],
) -> dict:
  """Cherche une commande dans le système de commandes."""
  return biclou.chercher_commande(numero)


@serveur.tool()
def verifier_stock(
  modele: Annotated[
    Literal[tuple(biclou.MODELES)], Field(description="le modèle de vélo")
  ],
  taille: ...,
  magasin: ...,
) -> dict:
  """Vérifie le stock d'un modèle de vélo dans un magasin."""
  return biclou.verifier_stock(modele, taille, magasin)


...

if __name__ == "__main__":
  serveur.run()
```

#### Solution

In [ ]:
%%writefile serveur_biclou.py
"""Le serveur MCP des outils de Biclou & Cie, en stdio."""

from typing import Annotated, Literal

import biclou
from mcp.server.mcpserver import MCPServer
from pydantic import Field

serveur = MCPServer("biclou", log_level="WARNING")


@serveur.tool()
def chercher_commande(
  numero: Annotated[
    str, Field(description="le numéro de commande, CMD- suivi de 6 chiffres")
  ],
) -> dict:
  """Cherche une commande dans le système de commandes."""
  return biclou.chercher_commande(numero)


@serveur.tool()
def verifier_stock(
  modele: Annotated[
    Literal[tuple(biclou.MODELES)], Field(description="le modèle de vélo")
  ],
  taille: Annotated[
    Literal[tuple(biclou.TAILLES)], Field(description="la taille de cadre")
  ],
  magasin: Annotated[
    Literal[tuple(biclou.MAGASINS)], Field(description="la ville du magasin")
  ],
) -> dict:
  """Vérifie le stock d'un modèle de vélo dans un magasin."""
  return biclou.verifier_stock(modele, taille, magasin)


@serveur.tool()
def chercher_politique(
  theme: Annotated[
    Literal[tuple(biclou.THEMES)],
    Field(description="la rubrique des règles où chercher"),
  ],
  question: Annotated[str, Field(description="la question, en quelques mots")],
) -> dict:
  """Cherche dans les règles du magasin les passages qui répondent à une question."""
  return biclou.chercher_politique(theme, question)


@serveur.tool()
def creer_ticket(
  categorie: Annotated[
    Literal[tuple(biclou.CATEGORIES)],
    Field(description="la catégorie de la demande"),
  ],
  priorite: Annotated[
    Literal[tuple(biclou.PRIORITES)], Field(description="la priorité de la demande")
  ],
  resume: Annotated[str, Field(description="la demande en une phrase")],
) -> dict:
  """Crée un ticket pour une demande que l'équipe doit traiter."""
  return biclou.creer_ticket(categorie, priorite, resume)


@serveur.tool()
def escalader(
  raison: Annotated[
    Literal[tuple(biclou.RAISONS)], Field(description="la raison du transfert")
  ],
  resume: Annotated[str, Field(description="la situation en une phrase")],
) -> dict:
  """Transmet la demande à un conseiller, pour un problème de sécurité ou d'argent."""
  return biclou.escalader(raison, resume)


if __name__ == "__main__":
  serveur.run()

### 2.2 Le client

Connectez-vous au serveur : `SERVEUR`, ses paramètres de lancement en stdio (le client lance `serveur_biclou.py` comme un processus, et lui parle par son entrée et sa sortie standard), puis `async with Client(SERVEUR) as mcp:` ; dans une cellule de notebook, `await` s'écrit directement. Listez ses outils (`list_tools`) et écrivez `vers_outil(outil)`, qui convertit un outil MCP au format des schémas qu'attend le *chat template* : `{"type": "function", "function": {"name": ..., "description": ..., "parameters": ...}}`, ses paramètres sans les clés `title` que le SDK peut y laisser (pydantic les tire du nom de chaque argument). Rangez les cinq dans `OUTILS`, les schémas des arguments tels quels dans `SCHEMAS` (par nom d'outil), et comparez chaque outil à celui d'`outils_reference` : quelles clés diffèrent ?

Puis écrivez `execute(mcp, appel)`, le résultat d'un appel (`{"name": ..., "arguments": {...}}`) par le serveur (`call_tool`) : le JSON de son texte, ou `{"erreur": <le texte>}` quand le serveur signale une erreur. Essayez-la sur les appels d'`essais` : un appel valide, une taille hors de la liste, une commande introuvable, un outil inconnu. Lesquels le serveur refuse-t-il avant d'exécuter la fonction ?

**Documentation utile**

- [Le client MCP](https://py.sdk.modelcontextprotocol.io/client/)
- [Les transports](https://py.sdk.modelcontextprotocol.io/client/transports/)

In [ ]:
essais = [
  dict(
    name="verifier_stock",
    arguments=dict(modele="Biclou Urbain 3", taille="S", magasin="Lille"),
  ),
  dict(
    name="verifier_stock",
    arguments=dict(modele="Biclou Urbain 3", taille="XXL", magasin="Lille"),
  ),
  dict(name="chercher_commande", arguments=dict(numero="CMD-000000")),
  dict(name="annuler_commande", arguments=dict(numero="CMD-103338")),
]

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `StdioServerParameters(command=sys.executable, args=["serveur_biclou.py"])` : le serveur, lancé par le Python du notebook
- `(await mcp.list_tools()).tools` : les outils, chacun avec `.name`, `.description` et `.input_schema`, le schéma de ses arguments
- `await mcp.call_tool(nom, arguments)` : le résultat, son texte dans `.content[0].text`, et `.is_error`, vrai quand le serveur a refusé l'appel ou que l'outil a échoué
- `sorted(k for k in a | b if a.get(k) != b.get(k))` : les clés de deux dictionnaires `a` et `b` dont les valeurs diffèrent
- Exemple à vérifier : le premier essai rend `{'modele': 'Biclou Urbain 3', 'taille': 'S', 'magasin': 'Lille', 'quantite': 2}`

```python
SERVEUR = StdioServerParameters(command=sys.executable, args=["serveur_biclou.py"])


def sans_titres(schema: dict) -> dict:
  proprietes = {
    nom: {cle: v for cle, v in p.items() if cle != "title"}
    for nom, p in schema["properties"].items()
  }
  sans_titre = {cle: v for cle, v in schema.items() if cle != "title"}
  return {**sans_titre, "properties": proprietes}


def vers_outil(outil) -> dict:
  fonction = dict(
    name=outil.name, description=..., parameters=sans_titres(outil.input_schema)
  )
  return dict(type="function", function=fonction)


async def execute(mcp, appel: dict) -> dict:
  resultat = await mcp.call_tool(appel["name"], appel["arguments"])
  texte = resultat.content[0].text
  return ...


async with Client(SERVEUR) as mcp:
  outils_mcp = (await mcp.list_tools()).tools
  OUTILS = [vers_outil(o) for o in outils_mcp]
  SCHEMAS = {o.name: o.input_schema for o in outils_mcp}
  references = {o["function"]["name"]: o["function"] for o in outils_reference}
  for outil in OUTILS:
    a, b = outil["function"], references[...]
    print(a["name"], ": diffèrent", sorted(k for k in a | b if a.get(k) != b.get(k)))
  for appel in essais:
    print(appel["name"], appel["arguments"], "->", await execute(mcp, appel))
```

#### Solution

In [ ]:
SERVEUR = StdioServerParameters(command=sys.executable, args=["serveur_biclou.py"])


def sans_titres(schema: dict) -> dict:
  proprietes = {
    nom: {cle: v for cle, v in p.items() if cle != "title"}
    for nom, p in schema["properties"].items()
  }
  sans_titre = {cle: v for cle, v in schema.items() if cle != "title"}
  return {**sans_titre, "properties": proprietes}


def vers_outil(outil) -> dict:
  """Un outil MCP au format des schémas du chat template."""
  fonction = dict(
    name=outil.name,
    description=outil.description,
    parameters=sans_titres(outil.input_schema),
  )
  return dict(type="function", function=fonction)


async def execute(mcp, appel: dict) -> dict:
  """Le résultat d'un appel par le serveur, ou l'erreur que l'agent recevra."""
  if not isinstance(appel["arguments"], dict):
    return {"erreur": "arguments invalides : un objet JSON attendu"}
  resultat = await mcp.call_tool(appel["name"], appel["arguments"])
  texte = resultat.content[0].text
  return {"erreur": texte} if resultat.is_error else json.loads(texte)


async with Client(SERVEUR) as mcp:
  outils_mcp = (await mcp.list_tools()).tools
  OUTILS = [vers_outil(o) for o in outils_mcp]
  SCHEMAS = {o.name: o.input_schema for o in outils_mcp}
  print(json.dumps(outils_mcp[1].input_schema["properties"]["taille"]))
  references = {o["function"]["name"]: o["function"] for o in outils_reference}
  for outil in OUTILS:
    a, b = outil["function"], references[outil["function"]["name"]]
    print(a["name"], ": diffèrent", sorted(k for k in a | b if a.get(k) != b.get(k)))
  print()
  for appel in essais:
    print(appel["name"], appel["arguments"], "->", await execute(mcp, appel))

# Seule `return` diffère : outils.json décrit aussi ce que rend chaque outil, ce que le
# serveur ne déclare pas pour une fonction qui rend un dict. Rien n'y perd : aucun
# serveur compatible avec l'API d'OpenAI, celle d'un agent en production, ne transmet
# cette clé au modèle.
# Le modèle voit ce que déclare le serveur : changer une description, c'est changer le
# serveur, pas l'agent. Un argument hors schéma est refusé avant la fonction ; une
# commande introuvable, elle, est une réponse de la fonction.

## 3. La boucle ReAct

ReAct (Yao et al., 2023) alterne une pensée, une action (un appel d'outil) et son observation (le résultat), jusqu'à la réponse. Écrivez la boucle de l'agent, `agent(mcp, messages)`, une fonction `async`, qui répète au plus `MAX_ETAPES` fois :

1. générer la suite de `messages` (`genere_tout`, avec les outils `OUTILS`, le cache `cache_de(messages[0]["content"])` et `max_tokens` tokens au plus, `PENSEE_MAX` de plus quand le modèle raisonne, avec `pense=True`), puis en séparer la pensée (`separe_pensee`) ;
2. lire ses appels (`lit_appels`) : sans appel, le texte est la réponse, et la boucle s'arrête ;
3. sinon, ajouter à `messages` un message `assistant` qui porte le texte hors des appels et les appels (`tool_calls`, au format d'`appel_chat`), puis, pour chaque appel, l'exécuter par le client MCP (`execute`) et ajouter son résultat, un message `tool` en texte JSON.

Une sortie illisible (un raisonnement inachevé, ou `lit_appels` qui rend `None`) ou le budget épuisé arrêtent la boucle sur la réponse de repli, `REPLI`, qui passe la main à un conseiller. La réponse, quelle qu'elle soit, est ajoutée à `messages` (un message `assistant`). La fonction rend le trajet : ses `appels`, si son format était lisible (`format_lisible`), son nombre de `generations` et sa `reponse`. Avec `bavard=True`, elle affiche la trace : la pensée, chaque action et son observation, puis la réponse.

Faites-la tourner sur les demandes de l'exercice 1, deux fois : sans raisonnement, comme l'agent du cours (`pense=False` : le bloc `<think>` de Qwen3 éteint, pour la latence), puis avec (`pense=True`), sur les `N_PENSEE` premières (une seule sans GPU, où la pensée prend plusieurs minutes par demande). Le petit modèle appelle-t-il ses outils, avec et sans pensée, et que coûte-t-elle en temps ? Quand il les appelle, que ne peut-il plus inventer, face aux réponses directes ? Que reste-t-il qu'aucun outil ne vérifie ?

In [ ]:
BLOC_APPEL = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.S)


def lit_appels(texte: str) -> list[dict] | None:
  """Les appels d'outil d'une sortie du modèle, None si un bloc est illisible."""
  appels = []
  for bloc in BLOC_APPEL.findall(texte):
    try:
      appel = json.loads(bloc)
      arguments = appel.get("arguments", {})
      if isinstance(arguments, str):
        arguments = json.loads(arguments)
    except (json.JSONDecodeError, AttributeError):
      return None
    if not isinstance(appel.get("name"), str):
      return None
    appels.append(dict(name=appel["name"], arguments=arguments))
  if texte.count("<tool_call>") != len(appels):  # un bloc pas refermé
    return None
  return appels


def appel_chat(appel: dict) -> dict:
  """Un appel au format des conversations : ses arguments en texte JSON."""
  arguments = json.dumps(appel["arguments"], ensure_ascii=False)
  return dict(type="function", function=dict(name=appel["name"], arguments=arguments))


def separe_pensee(texte: str) -> tuple[str, str | None]:
  """Le raisonnement d'une sortie (son bloc <think>, vide sans) et la suite ; None pour
  la suite quand le raisonnement est coupé avant sa fin."""
  if "</think>" in texte:
    pensee, suite = texte.split("</think>", 1)
    return pensee.replace("<think>", "").strip(), suite
  if "<think>" in texte:
    return texte.replace("<think>", "").strip(), None
  return "", texte


caches = {}


def cache_de(systeme: str):
  """Le cache du début des prompts d'une consigne, avec les outils, calculé une fois."""
  if systeme not in caches:
    caches[systeme] = prefixe(modele, tokenizer, systeme, OUTILS)
  return caches[systeme]


def conversation(message: str, systeme: str = CONSIGNES_AGENT) -> list[dict]:
  return [dict(role="system", content=systeme), dict(role="user", content=message)]


MAX_ETAPES = 4  # générations au plus : deux appels, une relance, la réponse
PENSEE_MAX = 512  # tokens de plus par génération pour le raisonnement (pense=True)
REPLI = """Bonjour,

Merci pour votre message. Nous n'avons pas pu traiter votre demande automatiquement : un conseiller de notre équipe la reprend dès maintenant et vous répondra par écrit dans les meilleurs délais, avec les informations de votre dossier. Vous n'avez rien à faire d'ici là.

Bonne journée,
L'équipe Biclou & Cie"""

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `genere_tout(modele, tokenizer, [messages], outils=OUTILS, cache=cache, max_tokens=max_tokens + PENSEE_MAX * pense, pense=pense)[0]` : la suite d'une seule conversation
- `pensee, suite = separe_pensee(texte)` : `suite` vaut `None` quand le raisonnement est coupé, une sortie illisible
- `nettoie(BLOC_APPEL.sub("", suite))` : le texte hors des appels, la réponse quand il n'y a pas d'appel
- Un appel : `dict(role="assistant", content=contenu, tool_calls=[appel_chat(a) for a in appels])`, puis son résultat : `dict(role="tool", content=json.dumps(observation, ensure_ascii=False))`
- `await execute(mcp, a)` : `execute` est une coroutine, d'où `async def agent` et `await agent(...)`
- Sans GPU, une génération prend de quelques secondes (un appel) à une minute (une réponse), et plus avec la pensée

```python
async def agent(mcp, messages, max_tokens=200, pense=False, bavard=False) -> dict:
  trajet = dict(appels=[], format_lisible=True, generations=0, reponse=REPLI)
  cache = cache_de(messages[0]["content"])
  for _ in range(MAX_ETAPES):
    texte = genere_tout(
      modele,
      tokenizer,
      [messages],
      outils=...,
      cache=cache,
      max_tokens=max_tokens + PENSEE_MAX * pense,
      pense=pense,
    )[0]
    trajet["generations"] += 1
    pensee, suite = separe_pensee(texte)
    appels = None if suite is None else lit_appels(suite)
    if appels is None:
      trajet["format_lisible"] = False
      break
    contenu = nettoie(BLOC_APPEL.sub("", suite))
    if bavard and pensee:
      print("Pensée :", " ".join(pensee.split()))
    if not appels:
      trajet["reponse"] = ...
      break
    messages.append(dict(role="assistant", content=contenu, tool_calls=...))
    for a in appels:
      observation = await execute(mcp, a)
      trajet["appels"].append(a)
      messages.append(dict(role="tool", content=...))
      if bavard:
        print("Action :", a["name"], a["arguments"])
        print("Observation :", observation)
  messages.append(dict(role="assistant", content=trajet["reponse"]))
  if bavard:
    print("Réponse :", trajet["reponse"])
  return trajet


traces = []
async with Client(SERVEUR) as mcp:
  for pense, lot in [(False, directes), (True, directes.head(N_PENSEE))]:
    debut = time.time()
    for d in lot.itertuples():
      print(f"\n=== {'Avec' if pense else 'Sans'} pensée : {d.message}\n")
      messages = conversation(d.message)
      traces.append((messages, await agent(mcp, messages, pense=..., bavard=True)))
    duree = time.time() - debut
    print(f"\n{len(lot)} demandes en {duree:.0f} s, pense={pense}")
```

### Solution

In [ ]:
async def agent(mcp, messages, max_tokens=200, pense=False, bavard=False) -> dict:
  """La boucle ReAct : génère, exécute les appels par le serveur MCP, recommence
  jusqu'à une réponse sans appel ou MAX_ETAPES générations. `messages` s'allonge des
  appels, de leurs résultats et de la réponse."""
  trajet = dict(appels=[], format_lisible=True, generations=0, reponse=REPLI)
  cache = cache_de(messages[0]["content"])
  for _ in range(MAX_ETAPES):
    texte = genere_tout(
      modele,
      tokenizer,
      [messages],
      outils=OUTILS,
      cache=cache,
      max_tokens=max_tokens + PENSEE_MAX * pense,
      pense=pense,
    )[0]
    trajet["generations"] += 1
    pensee, suite = separe_pensee(texte)
    appels = None if suite is None else lit_appels(suite)
    if appels is None:  # sortie illisible : repli
      trajet["format_lisible"] = False
      break
    contenu = nettoie(BLOC_APPEL.sub("", suite))
    if bavard and pensee:
      print("Pensée :", " ".join(pensee.split()))
    if not appels:  # la réponse
      trajet["reponse"] = contenu
      break
    messages.append(
      dict(
        role="assistant", content=contenu, tool_calls=[appel_chat(a) for a in appels]
      )
    )
    for a in appels:
      observation = await execute(mcp, a)
      trajet["appels"].append(a)
      messages.append(
        dict(role="tool", content=json.dumps(observation, ensure_ascii=False))
      )
      if bavard:
        print("Action :", a["name"], a["arguments"])
        print("Observation :", observation)
  messages.append(dict(role="assistant", content=trajet["reponse"]))
  if bavard:
    print("Réponse :", trajet["reponse"])
  return trajet


traces = []
async with Client(SERVEUR) as mcp:
  for pense, lot in [(False, directes), (True, directes.head(N_PENSEE))]:
    debut = time.time()
    for d in lot.itertuples():
      print(f"\n=== {'Avec' if pense else 'Sans'} pensée : {d.message}\n")
      messages = conversation(d.message)
      traces.append((messages, await agent(mcp, messages, pense=pense, bavard=True)))
    duree = time.time() - debut
    print(f"\n{len(lot)} demandes en {duree:.0f} s, pense={pense}")

# Un petit modèle sans pensée peut répondre sans chercher : c'est l'erreur que le TP
# suivant apprend à corriger, plutôt que de payer la pensée, des centaines de tokens
# par génération. Ce qu'il dit d'une commande qu'il n'a pas lue reste inventé.

## 4. Plan & Execute

Pour une demande qui en réunit deux, ReAct décide d'un appel à la fois, en relisant toute la conversation à chaque génération. Plan & Execute décompose d'abord : un planificateur écrit tout le plan, un exécutant le suit, puis le modèle répond. Ici, une étape du plan est un appel d'outil complet, son outil et ses arguments (`Etape`, `Plan`, cellule ci-dessous) : le plan se valide avant toute exécution, et s'exécute sans le modèle, d'où deux générations en tout, le plan et la réponse. La contrepartie : un argument ne peut pas dépendre du résultat d'une étape précédente. Une étape en langage naturel (« vérifier la commande, puis transférer si le colis est perdu ») le permettrait, mais il faudrait au modèle une génération par étape pour la traduire en appel, comme ReAct. Avec ces outils, les arguments vérifiables viennent du message ; seuls la priorité d'un ticket ou la raison d'un transfert dépendraient parfois de la fiche.

`composees` réunit, par une graine fixe, des paires de demandes de l'historique de scénarios différents, en un message, avec leurs appels attendus. Écrivez :

- `planifie(message)` : le plan du modèle (`CONSIGNES_PLAN` en message `system`, sans outils dans le *chat template* : la consigne les décrit, et 256 tokens au plus), le JSON de sa sortie (de la première accolade à la dernière) validé par `Plan` ; `None` pour une sortie qui n'est pas un plan valide ;
- `plan_execute(mcp, messages)`, qui fait comme `agent` mais planifie : le plan du dernier message, chaque étape exécutée par le serveur et ajoutée à `messages` comme dans la boucle (un message `assistant` qui porte l'appel, puis son résultat), puis une génération pour la réponse ; la réponse de repli si le plan est invalide. Elle rend le même trajet qu'`agent`.

Comparez les deux sur `composees`, sans pensée, comme l'agent du cours : les appels attendus trouvés (`trouves`), les appels faits, les générations et la durée, par méthode. Le planificateur ne passe pas par le format d'appel d'outils du *chat template* : écrit-il des appels que la boucle ReAct ne fait pas ? Sans GPU, le petit modèle appelle peu ses outils dans la boucle et écrit souvent un plan vide ou invalide : la comparaison vaut surtout sur GPU.

In [ ]:
simples = demandes[
  (demandes["canal"] == "chat")
  & (demandes["appels_attendus"].str.len() == 1)
  & ~demandes["message"].str.contains("\n")
]
hasard = random.Random(0)
composees = []
while len(composees) < N_COMPOSEES:
  a, b = simples.sample(2, random_state=hasard.randrange(10**6)).to_dict("records")
  if a["scenario"] == b["scenario"]:
    continue
  second = re.sub(r"^(Bonjour|Bjr|Salut)\W*", "", b["message"])
  composees.append(
    dict(
      message=f"{a['message']} Autre chose : {second}",
      appels_attendus=a["appels_attendus"] + b["appels_attendus"],
    )
  )
print(composees[0]["message"], "\n", composees[0]["appels_attendus"])

ARGUMENTS_LIBRES = {"question", "resume"}  # du texte, qu'on ne compare pas mot à mot


def exacts(appel: dict) -> dict:
  """Les arguments vérifiables d'un appel : sans les arguments libres."""
  arguments = appel["arguments"] if isinstance(appel["arguments"], dict) else {}
  return {a: v for a, v in arguments.items() if a not in ARGUMENTS_LIBRES}


def trouves(attendus: list[dict], obtenus: list[dict]) -> int:
  """Les appels attendus qu'on retrouve parmi les obtenus : même outil, mêmes
  arguments vérifiables."""
  restants = [(o["name"], exacts(o)) for o in obtenus]
  n = 0
  for a in attendus:
    if (a["name"], exacts(a)) in restants:
      restants.remove((a["name"], exacts(a)))
      n += 1
  return n


class Etape(BaseModel):
  outil: Literal[tuple(SCHEMAS)]
  arguments: dict


class Plan(BaseModel):
  etapes: list[Etape]


REGLES = CONSIGNES_AGENT.split("\n\n")[1]  # « Quand appeler un outil : »
CONSIGNES_PLAN = (
  "Tu planifies le traitement des messages de clients de Biclou & Cie : la liste des "
  "appels d'outils à faire, dans l'ordre, avant toute réponse. Les outils :\n"
  + "\n".join(json.dumps(o["function"], ensure_ascii=False) for o in OUTILS)
  + f"\n\n{REGLES}\n\n"
  + 'Réponds par le seul plan, en JSON sur une ligne : {"etapes": [{"outil": "...", '
  + '"arguments": {...}}]}, une étape par appel, et {"etapes": []} quand aucun outil '
  + "n'est utile."
)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `genere_tout(modele, tokenizer, [conversation(message, CONSIGNES_PLAN)], cache=cache_plan, max_tokens=256)[0]`, avec `cache_plan = prefixe(modele, tokenizer, CONSIGNES_PLAN)` calculé une fois : sans `outils`, comme le cache
- `texte[texte.find("{") : texte.rfind("}") + 1]` : le JSON de la sortie ; `Plan.model_validate_json(json_)` le valide, ou lève `ValidationError`
- Une étape devient l'appel `dict(name=etape.outil, arguments=etape.arguments)`
- Après les étapes, `genere_tout(..., outils=OUTILS, cache=cache_de(messages[0]["content"]), max_tokens=200)` : la réponse ; un appel qu'elle ferait encore est ôté (`BLOC_APPEL.sub`)
- Exemple à vérifier : `trouves([{"name": "chercher_commande", "arguments": {"numero": "CMD-103338"}}], [])` donne `0`

```python
cache_plan = prefixe(modele, tokenizer, CONSIGNES_PLAN)


def planifie(message: str) -> Plan | None:
  texte = genere_tout(
    modele,
    tokenizer,
    [conversation(message, CONSIGNES_PLAN)],
    cache=cache_plan,
    max_tokens=256,
  )[0]
  try:
    return Plan.model_validate_json(texte[texte.find("{") : texte.rfind("}") + 1])
  except ValidationError:
    return ...


async def plan_execute(mcp, messages) -> dict:
  trajet = dict(appels=[], format_lisible=True, generations=1, reponse=REPLI)
  plan = planifie(messages[-1]["content"])
  if plan is None:
    trajet["format_lisible"] = False
  else:
    for etape in plan.etapes:
      appel = dict(name=etape.outil, arguments=etape.arguments)
      messages.append(
        dict(role="assistant", content="", tool_calls=[appel_chat(appel)])
      )
      observation = await execute(mcp, appel)
      messages.append(dict(role="tool", content=...))
      trajet["appels"].append(appel)
    texte = genere_tout(
      modele,
      tokenizer,
      [messages],
      outils=OUTILS,
      cache=cache_de(messages[0]["content"]),
      max_tokens=200,
    )[0]
    trajet["generations"] += 1
    trajet["reponse"] = nettoie(BLOC_APPEL.sub("", texte)) or REPLI
  messages.append(dict(role="assistant", content=trajet["reponse"]))
  return trajet


lignes = []
fins = []  # les conversations, pour l'exercice 5
async with Client(SERVEUR) as mcp:
  for c in composees:
    for methode in ["ReAct", "Plan & Execute"]:
      debut = time.time()
      messages = conversation(c["message"])
      if methode == "ReAct":
        trajet = await agent(mcp, messages)
      else:
        trajet = await ...
      fins.append(messages)
      lignes.append(
        dict(
          methode=methode,
          attendus=len(c["appels_attendus"]),
          trouves=trouves(c["appels_attendus"], trajet["appels"]),
          appels=len(trajet["appels"]),
          generations=trajet["generations"],
          duree=time.time() - debut,
        )
      )
comparaison = pd.DataFrame(lignes)
comparaison.groupby("methode").sum().round()
```

### Solution

In [ ]:
cache_plan = prefixe(modele, tokenizer, CONSIGNES_PLAN)


def planifie(message: str) -> Plan | None:
  """Le plan du modèle pour un message, validé, ou None."""
  texte = genere_tout(
    modele,
    tokenizer,
    [conversation(message, CONSIGNES_PLAN)],
    cache=cache_plan,
    max_tokens=256,
  )[0]
  try:
    return Plan.model_validate_json(texte[texte.find("{") : texte.rfind("}") + 1])
  except ValidationError:
    return None


async def plan_execute(mcp, messages) -> dict:
  """Planifie, exécute le plan par le serveur MCP, puis répond : deux générations."""
  trajet = dict(appels=[], format_lisible=True, generations=1, reponse=REPLI)
  plan = planifie(messages[-1]["content"])
  if plan is None:  # un plan invalide : repli
    trajet["format_lisible"] = False
  else:
    for etape in plan.etapes:
      appel = dict(name=etape.outil, arguments=etape.arguments)
      messages.append(
        dict(role="assistant", content="", tool_calls=[appel_chat(appel)])
      )
      observation = await execute(mcp, appel)
      messages.append(
        dict(role="tool", content=json.dumps(observation, ensure_ascii=False))
      )
      trajet["appels"].append(appel)
    texte = genere_tout(
      modele,
      tokenizer,
      [messages],
      outils=OUTILS,
      cache=cache_de(messages[0]["content"]),
      max_tokens=200,
    )[0]
    trajet["generations"] += 1
    trajet["reponse"] = nettoie(BLOC_APPEL.sub("", texte)) or REPLI
  messages.append(dict(role="assistant", content=trajet["reponse"]))
  return trajet


lignes = []
fins = []  # les conversations, pour l'exercice 5
async with Client(SERVEUR) as mcp:
  for c in composees:
    for methode in ["ReAct", "Plan & Execute"]:
      debut = time.time()
      messages = conversation(c["message"])
      if methode == "ReAct":
        trajet = await agent(mcp, messages)
      else:
        trajet = await plan_execute(mcp, messages)
      fins.append(messages)
      lignes.append(
        dict(
          methode=methode,
          attendus=len(c["appels_attendus"]),
          trouves=trouves(c["appels_attendus"], trajet["appels"]),
          appels=len(trajet["appels"]),
          generations=trajet["generations"],
          duree=time.time() - debut,
        )
      )
      print(f"[{methode}] {[a['name'] for a in trajet['appels']]}")
comparaison = pd.DataFrame(lignes)
comparaison.groupby("methode").sum().round()

# Sur GPU, où l'agent appelle ses outils, le plan coûterait moins de générations : deux,
# quand ReAct en fait une par appel, plus la réponse. Sans GPU, ReAct, qui répond sans
# appeler, en fait moins. Et le plan est écrit avant d'avoir rien lu : une étape qu'il
# oublie n'est jamais rattrapée, quand ReAct peut encore appeler un outil.

## 5. Réflexion : relire et corriger

Réfléchir, pour un agent, c'est relire une sortie avant de la rendre, puis la corriger. La critique peut venir du modèle lui-même, d'un autre modèle (un juge, au dernier chapitre), ou du code, quand la règle se vérifie : ici, elle vient du code, qui vérifie les règles de forme de la charte. Écrivez `problemes_charte(reponse, message)`, la liste des règles qu'une réponse enfreint : elle ne commence pas par « Bonjour » (ou « Hello »), ne finit pas par la signature (`SIGNATURE`, ou celle de l'anglais ; le problème cite alors sa dernière ligne, pour que le modèle voie ce qu'il doit changer), tutoie le client (`TUTOIEMENT`), compte moins de 40 ou plus de 120 mots, ou ne rappelle pas le numéro de commande que cite le message ; une liste vide pour une réponse conforme. Puis `corrige(messages, problemes)` : un tour de plus, où le modèle reçoit ces problèmes (`CORRECTION`, en message `user` après sa réponse) et rend une nouvelle réponse.

Appliquez-les aux réponses des exercices 3 et 4 (`a_relire`, sauf les réponses de repli) : le taux de réponses conformes avant et après correction, et le coût, une génération de plus par réponse corrigée. Que corrige un tour, que laisse-t-il ?

In [ ]:
SIGNATURE = "Bonne journée,\nL'équipe Biclou & Cie"
TUTOIEMENT = re.compile(r"\b(tu|te|toi|ton|ta|tes)\b|\bt['’]", re.IGNORECASE)
CORRECTION = (
  "Contrôle avant envoi : votre réponse ne respecte pas la charte ({problemes}). "
  "Réécrivez-la en entier pour le client, en corrigeant ces points sans changer les "
  "faits."
)
a_relire = [m for m, _ in traces] + fins
a_relire = [m for m in a_relire if m[-1]["content"] != REPLI]
print(len(a_relire), "réponses à relire")

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `reponse.startswith(("Bonjour", "Hello"))`, `reponse.endswith((SIGNATURE, "The Biclou & Cie team"))`, `TUTOIEMENT.search(reponse)`, `len(reponse.split())` : les quatre premières règles
- `reponse.splitlines()[-1]` : la dernière ligne, que cite le problème de signature : le modèle voit ce qu'il a écrit, pas seulement la règle
- `re.findall(r"CMD-\d{6}", message)` : les numéros que le message cite, chacun à retrouver dans la réponse
- `CORRECTION.format(problemes="; ".join(problemes))` : le message du contrôle
- La correction est une génération sur la conversation allongée de ce message, avec les mêmes outils et le même cache que l'agent ; un appel qu'elle ferait est ôté (`BLOC_APPEL.sub`)
- Exemple à vérifier : `problemes_charte("Salut, tu l'as ?", "Ma commande CMD-103338 ?")` rend cinq problèmes

```python
def problemes_charte(reponse: str, message: str) -> list[str]:
  reponse = reponse.strip()
  problemes = []
  if not reponse.startswith(("Bonjour", "Hello")):
    problemes.append("elle ne commence pas par « Bonjour »")
  if not reponse.endswith(...):
    derniere = ...
    problemes.append(
      f"elle finit par « {derniere} », au lieu de « Bonne journée, » puis, seule "
      "sur la dernière ligne et sans point, « L'équipe Biclou & Cie »"
    )
  if TUTOIEMENT.search(reponse):
    problemes.append("elle tutoie le client")
  mots = len(reponse.split())
  if not 40 <= mots <= 120:
    problemes.append(f"elle fait {mots} mots, au lieu de 40 à 120")
  for numero in re.findall(r"CMD-\d{6}", message):
    if numero not in reponse:
      problemes.append(...)
  return problemes


def corrige(messages: list[dict], problemes: list[str]) -> str:
  controle = CORRECTION.format(problemes="; ".join(problemes))
  suite = messages + [dict(role="user", content=controle)]
  texte = genere_tout(
    modele,
    tokenizer,
    [suite],
    outils=OUTILS,
    cache=cache_de(messages[0]["content"]),
    max_tokens=200,
  )[0]
  return nettoie(BLOC_APPEL.sub("", texte))


debut = time.time()
avant, apres, corrections = [], [], 0
for messages in a_relire:
  message, reponse = messages[1]["content"], messages[-1]["content"]
  problemes = problemes_charte(reponse, message)
  avant.append(not problemes)
  if problemes:
    nouvelle = corrige(messages, problemes)
    corrections += 1
    restants = problemes_charte(nouvelle, message)
    print(problemes, "\n  ->", restants)
  else:
    restants = []
  apres.append(not restants)
print(f"Conformes : {sum(avant)} sur {len(avant)} avant, {sum(apres)} après")
print(f"{corrections} générations de plus, en {time.time() - debut:.0f} s")
```

### Solution

In [ ]:
def problemes_charte(reponse: str, message: str) -> list[str]:
  """Les règles de forme de la charte qu'une réponse enfreint."""
  reponse = reponse.strip()
  problemes = []
  if not reponse.startswith(("Bonjour", "Hello")):
    problemes.append("elle ne commence pas par « Bonjour »")
  if not reponse.endswith((SIGNATURE, "The Biclou & Cie team")):
    derniere = reponse.splitlines()[-1] if reponse else ""
    problemes.append(
      f"elle finit par « {derniere} », au lieu de « Bonne journée, » puis, seule "
      "sur la dernière ligne et sans point, « L'équipe Biclou & Cie »"
    )
  if TUTOIEMENT.search(reponse):
    problemes.append("elle tutoie le client")
  mots = len(reponse.split())
  if not 40 <= mots <= 120:
    problemes.append(f"elle fait {mots} mots, au lieu de 40 à 120")
  for numero in re.findall(r"CMD-\d{6}", message):
    if numero not in reponse:
      problemes.append(f"elle ne rappelle pas le numéro {numero}")
  return problemes


def corrige(messages: list[dict], problemes: list[str]) -> str:
  """Un tour de correction : le modèle reçoit les problèmes et réécrit sa réponse."""
  controle = CORRECTION.format(problemes="; ".join(problemes))
  suite = messages + [dict(role="user", content=controle)]
  texte = genere_tout(
    modele,
    tokenizer,
    [suite],
    outils=OUTILS,
    cache=cache_de(messages[0]["content"]),
    max_tokens=200,
  )[0]
  return nettoie(BLOC_APPEL.sub("", texte))


debut = time.time()
avant, apres, corrections = [], [], 0
for messages in a_relire:
  message, reponse = messages[1]["content"], messages[-1]["content"]
  problemes = problemes_charte(reponse, message)
  avant.append(not problemes)
  if problemes:
    nouvelle = corrige(messages, problemes)
    corrections += 1
    restants = problemes_charte(nouvelle, message)
    print(problemes, "\n  ->", restants)
  else:
    restants = []
  apres.append(not restants)
print(f"Conformes : {sum(avant)} sur {len(avant)} avant, {sum(apres)} après")
print(f"{corrections} générations de plus, en {time.time() - debut:.0f} s")

# La critique du code ne se trompe pas sur ce qu'elle vérifie, mais ne voit que la
# forme : un fait faux dans une réponse bien tournée passe ce contrôle.

## 6. État et mémoire

### 6.1 L'état d'une conversation

Le modèle ne garde rien d'une génération à l'autre : l'état d'une conversation, c'est la liste `messages`, qu'il relit en entier à chaque génération. La seconde cellule ci-dessous fait tourner l'agent sur `TOUR_1`, ajoute `TOUR_2`, qui y renvoie, à la même liste (`etat`, un message `user`), et le relance ; puis `TOUR_2` seul, dans une conversation neuve. Que fait l'agent du second message, avec et sans le premier ?

Cette liste s'allonge à chaque tour, résultats des outils compris, et chacun de ses tokens se paie à chaque génération. Écrivez `allege(messages)`, une copie de la conversation où chaque résultat d'outil perd ses champs superflus (`CHAMPS_SUPERFLUS` : ce qu'aucune réponse ne cite, comme le canal ou la date d'une commande, que l'historique ne cite jamais), et garde tout ce dont l'agent a besoin pour répondre : le statut et les dates d'une commande, la disponibilité d'un vélo, le texte d'une règle. Puis `nb_tokens(messages)`, les tokens que le modèle relit au tour suivant, et comparez-les avant et après `allege`, sur `etat` et sur `exemple`, une conversation de l'historique à deux appels (le message, les appels attendus, leurs résultats et la réponse) : sans GPU, l'agent appelle peu ses outils, et `etat` n'a souvent aucun résultat à alléger. Que gagne-t-on, et qu'est-ce qui pèse encore ?

In [ ]:
NUMERO = sorted(biclou.commandes.index[biclou.commandes["statut"] == "en_preparation"])[
  0
]
TOUR_1 = f"Bonjour, où en est ma commande {NUMERO} ? Je l'ai passée il y a deux jours."
TOUR_2 = "Je me suis trompé d'adresse de livraison. Il est encore temps de la changer ?"
print(TOUR_1, "\n", TOUR_2)

# Les champs d'un résultat d'outil qu'aucune réponse ne cite.
CHAMPS_SUPERFLUS = {"canal", "date_commande"}


def historique(demande) -> list[dict]:
  """Une conversation de l'historique : le message, les appels attendus, leurs
  résultats et la réponse."""
  messages = conversation(demande["message"])
  for appel, resultat in zip(demande["appels_attendus"], demande["resultats_outils"]):
    messages.append(dict(role="assistant", content="", tool_calls=[appel_chat(appel)]))
    messages.append(dict(role="tool", content=json.dumps(resultat, ensure_ascii=False)))
  messages.append(dict(role="assistant", content=demande["reponse"]))
  return messages


exemple = historique(
  demandes[demandes["classe"] == "chercher_commande+chercher_politique"].iloc[0]
)

In [ ]:
async with Client(SERVEUR) as mcp:
  etat = conversation(TOUR_1)
  print("--- Tour 1")
  await agent(mcp, etat, bavard=True)
  etat.append(dict(role="user", content=TOUR_2))
  print("\n--- Tour 2, à la suite du premier")
  await agent(mcp, etat, bavard=True)
  print("\n--- Tour 2, seul")
  await agent(mcp, conversation(TOUR_2), bavard=True)
print(f"\n{len(etat)} messages")

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `json.loads(m["content"])` pour un message `m` de rôle `tool` : son résultat ; `dict(m, content=...)` : une copie du message, son contenu remplacé, sans toucher à `messages`
- `{cle: v for cle, v in resultat.items() if cle not in CHAMPS_SUPERFLUS}` : le résultat sans ses champs superflus
- `len(tokenizer(tokenizer.apply_chat_template(messages, tools=OUTILS, tokenize=False)).input_ids)` : les tokens que le modèle relit, schémas des outils compris
- Exemple à vérifier : `allege(exemple)` a autant de messages qu'`exemple`, et son premier résultat d'outil, une fiche de commande, perd `canal` et `date_commande`

```python
def allege(messages: list[dict]) -> list[dict]:
  allegee = []
  for m in messages:
    if m["role"] == "tool":
      resultat = json.loads(m["content"])
      utile = {cle: v for cle, v in resultat.items() if ...}
      m = dict(m, content=json.dumps(utile, ensure_ascii=False))
    allegee.append(m)
  return allegee


def nb_tokens(messages: list[dict]) -> int:
  texte = tokenizer.apply_chat_template(messages, tools=OUTILS, tokenize=False)
  return len(...)


for nom, messages in [("etat", etat), ("exemple", exemple)]:
  avant, apres = nb_tokens(messages), nb_tokens(allege(messages))
  print(f"{nom} : {len(messages)} messages, {avant} tokens, {apres} allégée")
```

#### Solution

In [ ]:
def allege(messages: list[dict]) -> list[dict]:
  """Une copie de la conversation, chaque résultat d'outil sans ses champs
  superflus."""
  allegee = []
  for m in messages:
    if m["role"] == "tool":
      resultat = json.loads(m["content"])
      utile = {cle: v for cle, v in resultat.items() if cle not in CHAMPS_SUPERFLUS}
      m = dict(m, content=json.dumps(utile, ensure_ascii=False))
    allegee.append(m)
  return allegee


def nb_tokens(messages: list[dict]) -> int:
  """Les tokens que le modèle relit, schémas des outils compris."""
  texte = tokenizer.apply_chat_template(messages, tools=OUTILS, tokenize=False)
  return len(tokenizer(texte).input_ids)


for nom, messages in [("etat", etat), ("exemple", exemple)]:
  avant, apres = nb_tokens(messages), nb_tokens(allege(messages))
  print(f"{nom} : {len(messages)} messages, {avant} tokens, {apres} allégée")

# Seule la liste porte le premier tour : sans elle, le numéro de commande est perdu.
# Allégée, elle garde tout ce qui sert à répondre : le gain est faible, une vingtaine
# de tokens par fiche. Ce qui pèse : la consigne et les schémas des outils, relus à
# chaque génération (d'où le cache de leur début commun, `prefixe`), et, sur une
# longue conversation, les anciens tours, qu'on résumerait.

### 6.2 Une mémoire hors du modèle

Une conversation finie, sa liste `messages` est perdue. Une mémoire longue en garde quelques faits hors du modèle, ici un fichier JSON par client (`memoire/<client>.json`), relus au début de la conversation suivante, dans le message `system`. C'est le code qui choisit ce qu'on garde, pas le modèle : les numéros des commandes que le client a cités (dans ses messages) ou que l'agent a consultées, les tickets ouverts et les dossiers transmis, chacun avec sa date, tirés des messages du client et des résultats des outils (une fiche de commande a un `numero`, un ticket un `ticket`, un transfert un `dossier`).

Écrivez `retiens(client, messages)`, qui ajoute au fichier du client les faits d'une conversation, sans doublon, et oublie ceux de plus de `CONSERVATION_JOURS` jours, et `souvenirs(client)`, le texte à ajouter à la consigne de l'agent (une chaîne vide pour un client sans fichier). Retenez la conversation de 6.1 (`etat`), puis lancez l'agent sur `TOUR_3`, dans une conversation neuve dont la consigne est complétée des souvenirs du client : retrouve-t-il la commande ? Qu'est-ce qui ne doit pas entrer dans ce fichier, et pourquoi ?

**Documentation utile**

- [`pathlib.Path.read_text`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.read_text)
- [`datetime.timedelta`](https://docs.python.org/3/library/datetime.html#datetime.timedelta)

In [ ]:
MEMOIRE = Path("memoire")
MEMOIRE.mkdir(exist_ok=True)
CONSERVATION_JOURS = 90  # au-delà, un fait est oublié
CLIENT = "client-004217"  # l'identifiant du compte client, pas son nom
TOUR_3 = "Rebonjour, je vous ai écrit l'autre jour pour ma commande. Elle est partie ?"
FAITS = {"numero": "commande", "ticket": "ticket", "dossier": "dossier"}

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- Un fait : `{"date": "2026-11-18", "type": "commande", "valeur": "CMD-…"}` ; `date.today().isoformat()` : la date du jour en texte, `date.fromisoformat(texte)` l'inverse
- Les résultats des outils : `json.loads(m["content"])` pour chaque message `m` de rôle `tool` ; `FAITS` dit quelle clé d'un résultat donne quel type de fait ; `re.findall(r"CMD-\d{6}", m["content"])` pour un message `user` : les commandes qu'il cite
- `(MEMOIRE / f"{client}.json").exists()`, `.read_text()`, `.write_text(json.dumps(faits, ensure_ascii=False, indent=1))`
- `list({json.dumps(f, sort_keys=True): f for f in faits}.values())` : les faits sans doublon
- `conversation(TOUR_3, CONSIGNES_AGENT + "\n\n" + souvenirs(CLIENT))` : la conversation neuve ; sans GPU, `cache_de` calcule le cache de cette nouvelle consigne

```python
def retiens(client: str, messages: list[dict]) -> list[dict]:
  chemin = MEMOIRE / f"{client}.json"
  faits = json.loads(chemin.read_text()) if chemin.exists() else []
  aujourdhui = date.today()
  jour = aujourdhui.isoformat()
  for m in messages:
    if m["role"] == "user":
      for numero in re.findall(r"CMD-\d{6}", m["content"]):
        faits.append(dict(date=jour, type="commande", valeur=numero))
    if m["role"] != "tool":
      continue
    resultat = json.loads(m["content"])
    for cle, type_ in FAITS.items():
      if cle in resultat:
        faits.append(dict(date=..., type=type_, valeur=resultat[cle]))
  limite = aujourdhui - timedelta(days=CONSERVATION_JOURS)
  faits = [f for f in faits if date.fromisoformat(f["date"]) >= ...]
  faits = list({json.dumps(f, sort_keys=True): f for f in faits}.values())
  chemin.write_text(json.dumps(faits, ensure_ascii=False, indent=1))
  return faits


def souvenirs(client: str) -> str:
  chemin = MEMOIRE / f"{client}.json"
  if not chemin.exists():
    return ""
  lignes = [f"- {f['type']} {f['valeur']}, le {f['date']}" for f in ...]
  return "Ce que nos échanges précédents avec ce client ont enregistré :\n" + "\n".join(
    lignes
  )


print(retiens(CLIENT, etat))
systeme = CONSIGNES_AGENT + "\n\n" + souvenirs(CLIENT)
async with Client(SERVEUR) as mcp:
  await agent(mcp, conversation(TOUR_3, ...), bavard=True)
```

#### Solution

In [ ]:
def retiens(client: str, messages: list[dict]) -> list[dict]:
  """Ajoute à la mémoire du client les faits d'une conversation, et oublie les vieux."""
  chemin = MEMOIRE / f"{client}.json"
  faits = json.loads(chemin.read_text()) if chemin.exists() else []
  aujourdhui = date.today()
  jour = aujourdhui.isoformat()
  for m in messages:
    if m["role"] == "user":  # les commandes que le client cite
      for numero in re.findall(r"CMD-\d{6}", m["content"]):
        faits.append(dict(date=jour, type="commande", valeur=numero))
    if m["role"] != "tool":
      continue
    resultat = json.loads(m["content"])  # ce que les outils ont lu ou fait
    for cle, type_ in FAITS.items():
      if cle in resultat:
        faits.append(dict(date=jour, type=type_, valeur=resultat[cle]))
  limite = aujourdhui - timedelta(days=CONSERVATION_JOURS)
  faits = [f for f in faits if date.fromisoformat(f["date"]) >= limite]
  faits = list({json.dumps(f, sort_keys=True): f for f in faits}.values())
  chemin.write_text(json.dumps(faits, ensure_ascii=False, indent=1))
  return faits


def souvenirs(client: str) -> str:
  """Les faits retenus sur le client, à ajouter à la consigne de l'agent."""
  chemin = MEMOIRE / f"{client}.json"
  if not chemin.exists():
    return ""
  faits = json.loads(chemin.read_text())
  if not faits:
    return ""
  lignes = [f"- {f['type']} {f['valeur']}, le {f['date']}" for f in faits]
  return "Ce que nos échanges précédents avec ce client ont enregistré :\n" + "\n".join(
    lignes
  )


print(retiens(CLIENT, etat))
systeme = CONSIGNES_AGENT + "\n\n" + souvenirs(CLIENT)
print(souvenirs(CLIENT) or "Aucun souvenir", "\n")
async with Client(SERVEUR) as mcp:
  await agent(mcp, conversation(TOUR_3, systeme), bavard=True)

# On ne garde que des numéros et des dates : ni le nom, ni l'adresse, ni le texte des
# messages, des données personnelles que l'agent n'a pas à relire (le RGPD demande de
# les limiter au nécessaire et à une durée). Le statut d'une commande non plus : il
# change, l'agent relit la fiche.

## 7. Relever les erreurs de l'agent

Où l'agent du cours, sans pensée, se trompe-t-il ? La cellule ci-dessous le fait tourner sur `echantillon`, `N_PAR_CLASSE` demandes de l'historique par comportement attendu (`classe`), une seule sans GPU, et garde ses trajets ; la réponse finale est coupée à 96 tokens, on ne regarde ici que les appels. Sans GPU, attendez-vous à des appels manquants sur presque toutes les demandes à outil : le petit modèle sans pensée répond sans chercher, l'erreur que les deux TP suivants corrigent par l'affinage.

Écrivez `valide(appel)`, la liste des problèmes d'un appel face au schéma que déclare le serveur (`SCHEMAS`, par nom d'outil) : un outil inconnu, ou les erreurs que `jsonschema` trouve dans ses arguments ; une liste vide pour un appel valide. Puis `erreur_de(attendus, trajet)`, qui classe une demande, dans cet ordre :

1. `format` : une sortie illisible, ou un appel écrit dans le texte de la réponse (`APPEL_EN_TEXTE`, comme `<chercher_commande numero="CMD-…" />`), que la boucle ne lit pas, donc n'exécute pas ;
2. `appel_manquant` : les outils obtenus sont le début des outils attendus (aucun appel, ou le premier seulement) ;
3. `appel_inutile` : l'inverse, des appels de trop, comme un appel là où il n'en fallait aucun ;
4. `mauvais_outil` : d'autres outils que les attendus, un outil inconnu compris ;
5. `arguments_invalides` : les bons outils, mais un appel que `valide` refuse ;
6. `correct` : les bons outils, des arguments valides (leurs valeurs, le TP suivant les compare aux attendues).

Rangez l'erreur de chaque demande dans la colonne `erreur` d'`echantillon`, comptez-les, et croisez-les avec les comportements attendus : quelles erreurs dominent, sur quels outils ? Ce relevé est le point de départ du TP suivant, qui classe ces erreurs plus finement et en tire le jeu d'entraînement qui doit les corriger.

**Documentation utile**

- [`Validator.iter_errors`](https://python-jsonschema.readthedocs.io/en/stable/api/jsonschema/protocols/#jsonschema.protocols.Validator.iter_errors)
- [`pandas.crosstab`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)

In [ ]:
# Autant de demandes de chaque comportement attendu, au plus N_PAR_CLASSE.
echantillon = (
  demandes.sample(frac=1, random_state=0)
  .groupby("classe")
  .head(N_PAR_CLASSE)
  .reset_index(drop=True)
)
# Un appel écrit dans le texte de la réponse, hors des blocs <tool_call>.
APPEL_EN_TEXTE = re.compile(rf"<({'|'.join(SCHEMAS)})\b")
debut = time.time()
trajets = []
async with Client(SERVEUR) as mcp:
  for message in echantillon["message"]:
    trajets.append(await agent(mcp, conversation(message), max_tokens=96))
print(f"{len(echantillon)} demandes en {time.time() - debut:.0f} s")

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `jsonschema.Draft202012Validator(SCHEMAS[nom]).iter_errors(arguments)` : les erreurs, chacune avec son `.message`
- `noms_obtenus == noms_attendus[: len(noms_obtenus)]` : les outils obtenus sont le début des attendus (une liste vide aussi)
- `APPEL_EN_TEXTE.search(trajet["reponse"])` : un appel écrit dans le texte de la réponse, ou `None`
- `pd.crosstab(echantillon["classe"], echantillon["erreur"])` : le tableau des erreurs par comportement attendu
- Exemples à vérifier : `valide({"name": "verifier_stock", "arguments": {"modele": "Biclou Urbain 3", "taille": "XXL"}})` rend deux problèmes, la taille hors de la liste et le magasin manquant ; `erreur_de([], {"appels": [], "format_lisible": True, "reponse": REPLI})` rend `'correct'`

```python
def valide(appel: dict) -> list[str]:
  if appel["name"] not in SCHEMAS:
    return [f"outil inconnu : {appel['name']}"]
  validateur = jsonschema.Draft202012Validator(...)
  return [erreur.message for erreur in validateur.iter_errors(appel["arguments"])]


def erreur_de(attendus: list[dict], trajet: dict) -> str:
  noms_attendus = [a["name"] for a in attendus]
  noms_obtenus = [a["name"] for a in trajet["appels"]]
  if not trajet["format_lisible"] or ...:
    return "format"
  if noms_obtenus != noms_attendus:
    if noms_obtenus == noms_attendus[: len(noms_obtenus)]:
      return "appel_manquant"
    if ...:
      return "appel_inutile"
    return "mauvais_outil"
  if any(valide(a) for a in trajet["appels"]):
    return ...
  return "correct"


echantillon["erreur"] = [
  erreur_de(attendus, t) for attendus, t in zip(echantillon["appels_attendus"], trajets)
]
print(echantillon["erreur"].value_counts())
pd.crosstab(echantillon["classe"], echantillon["erreur"])
```

### Solution

In [ ]:
def valide(appel: dict) -> list[str]:
  """Les problèmes d'un appel face au schéma du serveur ; vide s'il est valide."""
  if appel["name"] not in SCHEMAS:
    return [f"outil inconnu : {appel['name']}"]
  validateur = jsonschema.Draft202012Validator(SCHEMAS[appel["name"]])
  return [erreur.message for erreur in validateur.iter_errors(appel["arguments"])]


def erreur_de(attendus: list[dict], trajet: dict) -> str:
  noms_attendus = [a["name"] for a in attendus]
  noms_obtenus = [a["name"] for a in trajet["appels"]]
  if not trajet["format_lisible"] or APPEL_EN_TEXTE.search(trajet["reponse"]):
    return "format"
  if noms_obtenus != noms_attendus:
    if noms_obtenus == noms_attendus[: len(noms_obtenus)]:
      return "appel_manquant"
    if noms_attendus == noms_obtenus[: len(noms_attendus)]:
      return "appel_inutile"
    return "mauvais_outil"
  if any(valide(a) for a in trajet["appels"]):
    return "arguments_invalides"
  return "correct"


echantillon["erreur"] = [
  erreur_de(attendus, t) for attendus, t in zip(echantillon["appels_attendus"], trajets)
]
print(echantillon["erreur"].value_counts())
invalides = [(a["name"], p) for t in trajets for a in t["appels"] for p in valide(a)]
print("\nAppels aux arguments invalides, par outil :")
print(pd.Series([nom for nom, _ in invalides], dtype=str).value_counts().to_dict())
for nom, probleme in invalides[:3]:
  print(f"  {nom} : {probleme}")
pd.crosstab(echantillon["classe"], echantillon["erreur"])

# Un argument inconnu passe : le schéma du serveur ne l'interdit pas
# (additionalProperties), et le serveur l'ignore.

## À retenir

- Un prompt seul répond avec ce que le modèle croit savoir : les faits d'une commande ne viennent que d'un outil.
- MCP sépare les outils de l'agent : le serveur les déclare (docstring, annotations, `Field`), valide les arguments et exécute ; le client liste et appelle. Le modèle voit ce que le serveur déclare.
- ReAct alterne pensée, action et observation, jusqu'à une réponse sans appel ou un budget d'étapes, avec une réponse de repli.
- Plan & Execute décompose avant d'agir : un plan validé, exécuté sans le modèle, moins de générations ; mais un plan écrit d'avance ne s'adapte pas à ce qu'il trouve.
- La réflexion relit avant de rendre : une critique par le code quand la règle se vérifie, et un tour de correction, qui coûte une génération.
- L'état d'une conversation est la liste de ses messages, relue à chaque génération, qu'on allège des résultats déjà lus ; une mémoire longue vit hors du modèle, choisie par le code, minimale et datée.
- Relever les erreurs de l'agent (format, outil, arguments, appel manquant ou inutile) dit quoi lui apprendre : c'est l'objet du jeu d'entraînement du TP suivant.